In [0]:
# 1. Define the parameters (Default value, Widget Name, UI Label)
dbutils.widgets.text("years", "", "Years of data to ingest") # There are various widgets to choose from like Multiselect,etc
# 2. Get the string value from the workflow
years_str = dbutils.widgets.get("years")
# 3. Convert it to a Python list using list comprehension and split()
years = [int(year.strip())  for year in years_str.split(",")]


In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install ../databricks_helpers
# dbutils.library.restartPython()

In [0]:
import os
import importlib
import fetch_bts_data

from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

#for reloading modules
#importlib.reload(fetch_bts_data)


bts_out_dir = f"{dbh.volume_directory()}/bts_data"
os.makedirs(bts_out_dir, exist_ok=True)
output_path = f"{dbh.volume_directory()}/daily_bts_data"
os.makedirs(output_path, exist_ok=True)
schemas_dir=f"{dbh.schemas_directory()}/daily_bts_data"
os.makedirs(schemas_dir, exist_ok=True)
checkpoints_dir=f"{dbh.checkpoints_directory()}/daily_bts_data"
os.makedirs(checkpoints_dir, exist_ok=True)

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS raw_data;
--SELECT current_catalog(), current_schema();

## Idempotent AutoDownloading of data from BTS site

In [0]:
# cleaning directory before adding files
# dbutils.fs.rm(bts_out_dir, True)
[fetch_bts_data.fetch_data_yearly(y, bts_out_dir) for y in years]

## Converting batch to daily data

In [0]:
import os
from pyspark.sql.functions import to_timestamp 




df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schemas_dir)
    .load(f"{dbh.volume_directory()}/bts_data/")
)


df = df.withColumn("FL_DATE", to_timestamp("FL_DATE", "M/d/yyyy h:mm:ss a"))\
 .withColumnRenamed("DAY_OF_MONTH","DAY")

(df.writeStream
    .format("parquet")
    .option("checkpointLocation", checkpoints_dir)
    .option("path", output_path)
    .option("header", "true")
    .partitionBy("YEAR", "MONTH", "DAY")
    .outputMode("append")
    .trigger(availableNow=True)   # Otherwise it will not stop
    .start()
)
